In [0]:
!git clone https://github.com/lvaleriu/ssd_keras-1.git

In [0]:
import os as os

os.chdir('/content/ssd_keras-1')

In [0]:
from keras_ssd300 import ssd_300

Using TensorFlow backend.


In [0]:
from keras import backend as K
from keras.models import load_model
from keras.preprocessing import image
from keras.optimizers import Adam
from imageio import imread
import numpy as np
from matplotlib import pyplot as plt

from keras_ssd300 import ssd_300
from keras_ssd_loss import SSDLoss
from keras_layer_AnchorBoxes import AnchorBoxes
from keras_layer_L2Normalization import L2Normalization
from ssd_box_encode_decode_utils import decode_y, decode_y2
from ssd_batch_generator import BatchGenerator

K.clear_session() # Clear previous models from memory.

model = ssd_300(image_size=(300, 300, 3),
                n_classes=90,
                l2_regularization=0.0005,
                scales=[0.1, 0.2, 0.37, 0.54, 0.71, 0.88, 1.05], # The scales for MS COCO are [0.07, 0.15, 0.33, 0.51, 0.69, 0.87, 1.05]
                aspect_ratios_per_layer=[[1.0, 2.0, 0.5],
                                         [1.0, 2.0, 0.5, 3.0, 1.0/3.0],
                                         [1.0, 2.0, 0.5, 3.0, 1.0/3.0],
                                         [1.0, 2.0, 0.5, 3.0, 1.0/3.0],
                                         [1.0, 2.0, 0.5],
                                         [1.0, 2.0, 0.5]],
                two_boxes_for_ar1=True,
                steps=[8, 16, 32, 64, 100, 300],
                offsets=[0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
                limit_boxes=False,
                variances=[0.1, 0.1, 0.2, 0.2],
                coords='centroids',
                normalize_coords=True,
                subtract_mean=[123, 117, 104],
                swap_channels=True)

W0710 07:47:05.407749 139694464419712 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:95: The name tf.reset_default_graph is deprecated. Please use tf.compat.v1.reset_default_graph instead.

W0710 07:47:05.410191 139694464419712 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:98: The name tf.placeholder_with_default is deprecated. Please use tf.compat.v1.placeholder_with_default instead.

W0710 07:47:05.463739 139694464419712 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:102: The name tf.get_default_graph is deprecated. Please use tf.compat.v1.get_default_graph instead.

W0710 07:47:05.466457 139694464419712 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:517: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.

W071

In [0]:
model.summary()

__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            (None, 300, 300, 3)  0                                            
__________________________________________________________________________________________________
idendity_layer (Lambda)         (None, 300, 300, 3)  0           input_1[0][0]                    
__________________________________________________________________________________________________
input_mean_normalization (Lambd (None, 300, 300, 3)  0           idendity_layer[0][0]             
__________________________________________________________________________________________________
input_channel_swap (Lambda)     (None, 300, 300, 3)  0           input_mean_normalization[0][0]   
__________________________________________________________________________________________________
conv1_1 (C

In [0]:
from keras.utils import plot_model
plot_model(model, to_file='model.png')

In [0]:
!wget --header="Host: doc-14-c8-docs.googleusercontent.com" --header="User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/75.0.3770.100 Safari/537.36" --header="Accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3" --header="Accept-Language: en-US,en;q=0.9" --header="Referer: https://drive.google.com/a/iiita.ac.in/uc?id=1vmEF7FUsWfHquXyCqO17UaXOPpRbwsdj&export=download" --header="Cookie: AUTH_4lgch8kproorp87c5erpsi9rqpbk46dg_nonce=4i9mgi54c1cce" --header="Connection: keep-alive" "https://doc-14-c8-docs.googleusercontent.com/docs/securesc/0ga3her86at4fpdr2b9n3spfq1b4g359/jnnan6nq3nrp6pfah014o8djijkdeu3q/1562724000000/01021765827329596762/14696292948499217652/1vmEF7FUsWfHquXyCqO17UaXOPpRbwsdj?e=download&h=06864318269717761326&nonce=4i9mgi54c1cce&user=14696292948499217652&hash=tegqulknt91v969af8ufv4me38oa9uei" -O "VGG_coco_SSD_300x300_iter_400000.h5" -c

In [0]:
# 1: Build the Keras model

K.clear_session() # Clear previous models from memory.

model = ssd_300(image_size=(300, 300, 3),
                n_classes=80,
                l2_regularization=0.0005,
                scales=[0.07, 0.15, 0.33, 0.51, 0.69, 0.87, 1.05], # The scales for MS COCO are 
                aspect_ratios_per_layer=[[1.0, 2.0, 0.5],
                                         [1.0, 2.0, 0.5, 3.0, 1.0/3.0],
                                         [1.0, 2.0, 0.5, 3.0, 1.0/3.0],
                                         [1.0, 2.0, 0.5, 3.0, 1.0/3.0],
                                         [1.0, 2.0, 0.5],
                                         [1.0, 2.0, 0.5]],
                two_boxes_for_ar1=True,
                steps=[8, 16, 32, 64, 100, 300],
                offsets=[0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
                limit_boxes=False,
                variances=[0.1, 0.1, 0.2, 0.2],
                coords='centroids',
                normalize_coords=True,
                subtract_mean=[123, 117, 104],
                swap_channels=True)

# 2: Load the trained weights into the model.

# TODO: Set the path of the trained weights.

weights_path = '/content/ssd_keras-1/VGG_coco_SSD_300x300_iter_400000.h5'

model.load_weights(weights_path, by_name=True)

# 3: Compile the model so that Keras won't complain the next time you load it.

adam = Adam(lr=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-08, decay=5e-04)

ssd_loss = SSDLoss(neg_pos_ratio=3, n_neg_min=0, alpha=1.0)

model.compile(optimizer=adam, loss=ssd_loss.compute_loss)

W0710 07:47:14.239746 139694464419712 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/optimizers.py:790: The name tf.train.Optimizer is deprecated. Please use tf.compat.v1.train.Optimizer instead.

W0710 07:47:14.269625 139694464419712 deprecation.py:323] From /content/ssd_keras-1/keras_ssd_loss.py:133: to_float (from tensorflow.python.ops.math_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use `tf.cast` instead.
W0710 07:47:14.291672 139694464419712 deprecation.py:323] From /content/ssd_keras-1/keras_ssd_loss.py:74: add_dispatch_support.<locals>.wrapper (from tensorflow.python.ops.array_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
W0710 07:47:14.317956 139694464419712 deprecation.py:323] From /content/ssd_keras-1/keras_ssd_loss.py:166: to_int32 (from tensorflow.python.ops.math_ops) is deprecated and will be r

**Uncertainty Model**

In [0]:
!wget --header="Host: doc-08-9c-docs.googleusercontent.com" --header="User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/75.0.3770.100 Safari/537.36" --header="Accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3" --header="Accept-Language: en-US,en;q=0.9" --header="Referer: https://drive.google.com/drive/u/0/search?q=images" --header="Cookie: AUTH_4lgch8kproorp87c5erpsi9rqpbk46dg=14696292948499217652|1562724000000|2cld2sphg6invdh2r0n8ggvh60aqdv15" --header="Connection: keep-alive" "https://doc-08-9c-docs.googleusercontent.com/docs/securesc/0ga3her86at4fpdr2b9n3spfq1b4g359/m59icukh97cbl8rs6la0d085ira6o2hj/1562724000000/14696292948499217652/14696292948499217652/1kK5B_MnBHxFBG-Ys3UqVHuLP3QIdV137?h=06864318269717761326&e=download" -O "images.zip" -c

In [0]:
!unzip 'images.zip'

In [0]:
from tqdm import tqdm

image_names = [] # Store the images here.
input_images = [] # Store resized versions of the images here.

# We'll only load one image in this example.
imgs_path = '/content/ssd_keras-1/images/train/'

for i in tqdm(os.listdir(imgs_path)[:]):
  img_path = imgs_path+i
  image_names.append(i)

  img = image.load_img(img_path, target_size=(300, 300))
  img = image.img_to_array(img) 
  input_images.append(img)

input_images = np.array(input_images)

100%|██████████| 2977/2977 [00:23<00:00, 124.98it/s]


In [0]:
from keras import Model

uncertainty_model = Model(model.input, model.get_layer('fc7').output)
pred = uncertainty_model.predict(input_images)
print(pred.shape)

In [0]:
!mkdir 'activations_val'

In [0]:
for i in tqdm(range(len(image_names))):
  np.save('./activations_val/'+image_names[i][:-4]+'.npy', pred[i])

In [0]:
from keras import Model

uncertainty_model = Model(model.input, model.get_layer('conv2_1').output)
pred = uncertainty_model.predict(input_images[:300])
print(pred.shape)

(300, 150, 150, 128)


In [0]:
!mkdir 'activations_conv1_2'

In [0]:
!zip -r 'activations_val.zip' 'activations_val'

In [0]:
pred = model.predict(input_images)
print(pred.shape)

In [0]:
y_pred_decoded = decode_y(pred,
                          confidence_thresh=0.1,
                          iou_threshold=1.0,
                          top_k=9000,
                          input_coords='centroids',
                          normalize_coords=True,
                          img_height=300,
                          img_width=300)

np.set_printoptions(precision=2, suppress=True, linewidth=90)
print("Predicted boxes:\n")
print('    class    conf  xmin     ymin   xmax    ymax')
print(y_pred_decoded[0])

In [0]:
!mkdir 'all_boxes_cow_dataset_val'

In [0]:
for i in tqdm(range(len(image_names))):
  f = open('./all_boxes_cow_dataset_val/'+image_names[i][:-4]+'.txt', 'w')
  for j in range(len(y_pred_decoded[i])):
    f.write(str(y_pred_decoded[i][j][0])+' ')
    f.write(str(y_pred_decoded[i][j][1])+' ')
    f.write(str(y_pred_decoded[i][j][2])+' ')
    f.write(str(y_pred_decoded[i][j][3])+' ')
    f.write(str(y_pred_decoded[i][j][4])+' ')
    f.write(str(y_pred_decoded[i][j][5])+'\n')
  f.close()

In [0]:
!zip -r 'all_boxes_cow_dataset_val.zip' 'all_boxes_cow_dataset_val'

In [0]:
# Display the image and draw the predicted boxes onto it.
img_height = img_width = 300

# Set the colors for the bounding boxes
colors = plt.cm.hsv(np.linspace(0, 1, 21)).tolist()
classes = ['background',
           'aeroplane', 'bicycle', 'bird', 'boat',
           'bottle', 'bus', 'car', 'cat',
           'chair', 'cow', 'diningtable', 'dog',
           'horse', 'motorbike', 'person', 'pottedplant',
           'sheep', 'sofa', 'train', 'tvmonitor']

orig_images = []
orig_images.append(imread(imgs_path+image_names[2692]))

plt.figure(figsize=(20,12))
plt.imshow(orig_images[0])

current_axis = plt.gca()

for box in y_pred_decoded[0]:
    # Transform the predicted bounding boxes for the 300x300 image to the original image dimensions.
    xmin = box[-4] * orig_images[0].shape[1] / img_width
    ymin = box[-3] * orig_images[0].shape[0] / img_height
    xmax = box[-2] * orig_images[0].shape[1] / img_width
    ymax = box[-1] * orig_images[0].shape[0] / img_height
    color = colors[int(box[0])]
    label = '{}: {:.2f}'.format(classes[int(box[0])], box[1])
    current_axis.add_patch(plt.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin, color=color, fill=False, linewidth=2))  
    current_axis.text(xmin, ymin, label, size='x-large', color='white', bbox={'facecolor':color, 'alpha':1.0})

In [0]:
print(image_names[2692])

In [0]:
from google.colab import auth
from googleapiclient.http import MediaFileUpload
from googleapiclient.discovery import build

auth.authenticate_user()

In [0]:
drive_service = build('drive', 'v3')

def save_file_to_drive(name, path):
  file_metadata = {
    'name': name,
    'mimeType': 'application/octet-stream'
   }

  media = MediaFileUpload(path, 
                  mimetype='application/octet-stream',
                  resumable=True)

  created = drive_service.files().create(body=file_metadata,
                                 media_body=media,
                                 fields='id').execute()

  print('File ID: {}'.format(created.get('id')))

  return created

In [0]:
from shutil import copy

copy('./activations.zip', '/content/gdrive/My Drive/activations.zip')